In [1]:

import pandas as pd
import sqlite3

from IPython.display import display
from tqdm.auto import tqdm

from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import MODELS, DB_PATH
from src.model_selection import get_checkpoints
from src.embedding_selection.widgets import create_model_selector
from src.embedding_selection.load import create_pipeline
from src.embedding_selection.evaluate import evaluate_embedding

In [2]:
DINO_DIR = MODELS / "dino_fine_tune"
CATHEAD_DIR = MODELS / "category_head"
ANOMADAP_DIR = MODELS / "dino_adapter_block"
ANOMHEAD_DIR = MODELS / "anomaly_head"

dino_checkpoints = get_checkpoints(DINO_DIR)
cathead_checkpoints = get_checkpoints(CATHEAD_DIR)
anomadap_checkpoints = get_checkpoints(ANOMADAP_DIR)
anomhead_checkpoints = get_checkpoints(ANOMHEAD_DIR)

Found 11 checkpoints at /home/ciaran/Projects/bursary/UL-Summer-Bursary-2026/models/dino_fine_tune
Found 4 checkpoints at /home/ciaran/Projects/bursary/UL-Summer-Bursary-2026/models/category_head
Found 50 checkpoints at /home/ciaran/Projects/bursary/UL-Summer-Bursary-2026/models/dino_adapter_block
Found 47 checkpoints at /home/ciaran/Projects/bursary/UL-Summer-Bursary-2026/models/anomaly_head


In [ ]:
selector = create_model_selector(
    adapter_block_checkpoints=anomadap_checkpoints,
    anomaly_head_checkpoints=anomhead_checkpoints,
    fine_tuned_dino_checkpoints=dino_checkpoints,
    category_head_checkpoints=cathead_checkpoints
)

selector.display()

RadioButtons(description='Model Types:', options=('All Models', 'Anomaly Detection Models', 'Category Head Mod…

Dropdown(description='Model:', layout=Layout(width='800px'), options=(('Select a model', 'UNSELECTED'), ('2026…

Button(button_style='success', description='confirm_selection', layout=Layout(width='180px'), style=ButtonStyl…

Output()

In [5]:
selected_path = selector.get_selected_path()

pipeline = create_pipeline(selected_path)

In [6]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("""
    SELECT category, type, label, split
    FROM meta
""", conn)

conn.close()

meta = meta.drop(meta.index[pipeline.neg_indices]).reset_index(drop=True)

In [7]:
comparison: pd.DataFrame | None = None
prev_name: str | None = None

for stage, embedding in tqdm(pipeline, desc="Evaluating Embedding", position=0):
    stage_result = evaluate_embedding(embedding, meta).rename(columns={"auroc": stage.name})

    if comparison is None:
        comparison = stage_result
    else:
        comparison = comparison.merge(
            stage_result,
            on="category",
            how="inner"
        )

        comparison[f"{stage.name}_delta"] = (
            comparison[stage.name]
            - comparison[prev_name]
        )

    prev_name = stage.name

if comparison is None:
    raise RuntimeError("No stages found in pipeline")

comparison = comparison.sort_values("category").reset_index(drop=True)

Evaluating Embedding:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating Category:   0%|          | 0/15 [00:00<?, ?it/s]

Evaluating Category:   0%|          | 0/15 [00:00<?, ?it/s]

In [8]:
if comparison is None:
    raise RuntimeError(
        "Pipeline produced no embedding stages."
    )

display(
    comparison.style
    .hide(axis="index")
    .format(
        {
            column: "{:.4f}"
            for column in comparison.columns
            if column != "category"
        }
    )
)

category,dino,dino_adapter_block,dino_adapter_block_delta
bottle,1.0000,1.0000,0.0000
cable,0.9455,0.9430,-0.0024
capsule,0.9469,0.9481,0.0012
carpet,0.9852,0.9916,0.0064
grid,0.9816,0.9875,0.0058
hazelnut,0.9683,0.9926,0.0244
leather,1.0000,1.0000,0.0000
metal_nut,0.9829,0.9990,0.0161
pill,0.9504,0.9585,0.0082
screw,0.8225,0.9096,0.0871


In [9]:
comparison.mean(numeric_only=True)

dino                        0.961628
dino_adapter_block          0.977864
dino_adapter_block_delta    0.016236
dtype: float64

In [10]:
(comparison.iloc[:, -1:] >= 0).sum()

dino_adapter_block_delta    13
dtype: int64